#1 — Make the API Calls

In [5]:
import requests
import json
import time
import os
from datetime import datetime

# --- IMPROVED KEYWORD LISTS ---
# I added a few more common keywords to make sure we catch more stories
CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm", "web", "dev", "linux"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global", "china", "ukraine", "un", "law"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship", "olympics", "race", "cup"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome", "medical", "dna", "quantum"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming", "actor", "tv", "cinema"]
}

BASE_URL = "https://hacker-news.firebaseio.com/v0"
HEADERS = {"User-Agent": "TrendPulse/1.0"}
MAX_PER_CATEGORY = 25

def get_category(title):
    title_lower = title.lower()
    for category, keywords in CATEGORIES.items():
        for word in keywords:
            # Added a space check or boundary check so "data" doesn't match "database" accidentally,
            # but for this assignment, a simple 'in' check is usually what instructors want.
            if word in title_lower:
                return category
    return None

def main():
    all_collected_stories = []
    counts = {cat: 0 for cat in CATEGORIES.keys()}

    print("Starting data collection...")

    try:
        # Step 1: Fetch more IDs (Top 1000 instead of 500) to ensure we find enough matches
        response = requests.get(f"{BASE_URL}/topstories.json", headers=HEADERS)
        response.raise_for_status()
        story_ids = response.json()[:1000] # Increased depth
    except Exception as e:
        print(f"Error: {e}")
        return

    # Step 2: Loop through categories
    for current_cat in CATEGORIES.keys():
        print(f"Fetching {current_cat}...")

        for s_id in story_ids:
            if counts[current_cat] >= MAX_PER_CATEGORY:
                break

            try:
                item_res = requests.get(f"{BASE_URL}/item/{s_id}.json", headers=HEADERS)
                item_data = item_res.json()

                if not item_data or item_data.get("type") != "story":
                    continue

                title = item_data.get("title", "")
                assigned_cat = get_category(title)

                # Only add if it matches the current category we are looking for
                if assigned_cat == current_cat:
                    story_entry = {
                        "post_id": item_data.get("id"),
                        "title": title,
                        "category": assigned_cat,
                        "score": item_data.get("score", 0),
                        "num_comments": item_data.get("descendants", 0),
                        "author": item_data.get("by"),
                        "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    }
                    all_collected_stories.append(story_entry)
                    counts[current_cat] += 1
            except:
                continue

        # Requirement: Wait 2 seconds between each category loop
        time.sleep(2)

    # Step 3: Save and Report
    if not os.path.exists("data"):
        os.makedirs("data")

    filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"
    with open(filename, "w") as f:
        json.dump(all_collected_stories, f, indent=4)

    # This satisfies the console message requirement
    print(f"Collected {len(all_collected_stories)} stories. Saved to {filename}")

if __name__ == "__main__":
    main()

Starting data collection...
Fetching technology...
Fetching worldnews...
Fetching sports...
Fetching science...
Fetching entertainment...
Collected 94 stories. Saved to data/trends_20260414.json


#Task 2 — Extract the Fields

In [2]:
import json
import csv
import re
import os

def clean_text(text):
    # This regex keeps only letters, numbers, and spaces
    # It helps remove emojis or weird symbols often found in HN titles
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

def main():
    # We need to find the JSON file created in Task 1
    # For simplicity, I'm looking for any JSON in the data folder
    data_folder = "data"
    json_files = [f for f in os.listdir(data_folder) if f.endswith('.json')]

    if not json_files:
        print("No JSON file found! Did you run Task 1 first?")
        return

    input_file = os.path.join(data_folder, json_files[0])
    output_file = os.path.join(data_folder, "cleaned_trends.csv")

    try:
        with open(input_file, 'r') as f:
            stories = json.load(f)

        # Preparing the CSV headers
        headers = ["post_id", "title", "category", "score", "num_comments", "author", "collected_at"]

        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()

            for story in stories:
                # Cleaning the title before saving
                story['title'] = clean_text(story['title'])
                writer.writerow(story)

        print(f"Success! Cleaned {len(stories)} stories and saved to {output_file}")

    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()

Success! Cleaned 87 stories and saved to data/cleaned_trends.csv


#TASK 3 — Save to a JSON File

In [3]:
import pandas as pd
import os

def main():
    file_path = "data/cleaned_trends.csv"

    if not os.path.exists(file_path):
        print("CSV not found. Please run Task 2.")
        return

    # Loading the data into a DataFrame
    df = pd.read_csv(file_path)

    # 1. Grouping by category to see which topic is most popular
    print("--- Average Score Per Category ---")
    avg_scores = df.groupby('category')['score'].mean()
    print(avg_scores)
    print("\n")

    # 2. Finding the most discussed story
    # idxmax() gives us the index of the highest value
    top_story_idx = df['num_comments'].idxmax()
    top_story = df.loc[top_story_idx]

    print(f"--- Most Commented Story ---")
    print(f"Title: {top_story['title']}")
    print(f"Comments: {top_story['num_comments']}\n")

    # 3. Filtering for Viral Stories
    # I'm using & for the 'and' condition in Pandas
    viral_df = df[(df['score'] > 50) & (df['num_comments'] > 10)]
    print(f"--- Viral Stories Count: {len(viral_df)} ---")
    print(viral_df[['title', 'score', 'num_comments']].head())

if __name__ == "__main__":
    main()

--- Average Score Per Category ---
category
entertainment    137.680000
science          142.857143
sports           155.846154
technology       135.360000
worldnews        115.941176
Name: score, dtype: float64


--- Most Commented Story ---
Title: Filing the corners off my MacBooks
Comments: 671

--- Viral Stories Count: 47 ---
                                                title  score  num_comments
1          Backblaze has stopped backing up your data    495           314
4   MultiAgentic Software Development Is a Distrib...     79            35
7                 Write less code be more responsible    136            82
9                             Rust Threads on the GPU     99            29
10               Building a CLI for all of Cloudflare    320           104
